# fact_judge 점검
합성 답변과 가짜 클라이언트로 규칙 판정 및 LLM 판정 흐름을 확인합니다. 실제 API를 호출하지 않습니다.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import types

CHAT_DIR = Path.cwd() / 'src/card_benefit/payment_card/file_pipeline/chat'
CACHE_PATH = Path('/tmp/fact-judge-notebook-cache.json')
sys.path.insert(0, str(CHAT_DIR))
spec = importlib.util.spec_from_file_location('fact_judge_notebook', CHAT_DIR / 'fact_judge.py')
fact_judge = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fact_judge)

## 합성 입력 → 판정 → 결과 표시

In [ ]:
class FakeClient:
    def __init__(self):
        self.chat = types.SimpleNamespace(completions=types.SimpleNamespace(create=self.create))
        self.calls = 0
    def create(self, **kwargs):
        self.calls += 1
        assert 'temperature' not in kwargs and kwargs['max_completion_tokens'] >= 2000
        response = types.SimpleNamespace(choices=[types.SimpleNamespace(message=types.SimpleNamespace(content='{\"verdict\":\"correct\",\"reason\":\"뜻이 같습니다\"}'))], usage=types.SimpleNamespace(prompt_tokens=12, completion_tokens=8))
        return response

rows = [{
    'id': 'synthetic-1',
    'answer': '정기적으로 납부하는 결제에 혜택이 적용됩니다.',
    'question': {'question': '어떤 결제가 적용되나요?', 'expected': ['정기 납부 결제에 혜택 적용'], 'issuer': 'sample', 'type': 'condition', 'no_answer': False},
}]
fake = FakeClient()
judged, cost = fact_judge.judge_records(rows, client=fake, cache_path=CACHE_PATH, max_cost_usd=0.05, sleep=0)
print(judged)
print(fact_judge.summarize(rows, judged))
print({'fake_calls': fake.calls, **cost})